# Gemma 3 SAE Steering — Fixed Kaggle Launcher

Fixed launcher cell that syncs the repository and delegates execution to `main.py`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

project_root = Path('/kaggle/working/phase1_sae_steering')
entrypoint = project_root / 'main.py'
if not entrypoint.is_file():
    subprocess.run(
        ['git', 'clone', '--depth', '1', 'https://github.com/dddddsz/phase1_sae_steering.git', str(project_root)],
        check=True,
    )
else:
    subprocess.run(['git', '-C', str(project_root), 'pull', '--ff-only'], check=True)

if not entrypoint.is_file():
    raise FileNotFoundError(f'Project entrypoint not found: {entrypoint}')

print('Project root:', project_root)
print('Entrypoint:', entrypoint)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-q', '-r', str(project_root / 'requirements.txt')],
    check=True,
)

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU is not available; enable a GPU accelerator.')
print('CUDA devices:', torch.cuda.device_count())
print('Using cuda:0:', torch.cuda.get_device_name(0))

if not os.environ.get('HF_TOKEN'):
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')

subprocess.run([sys.executable, str(entrypoint)], check=True)